## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
from condpred import report
from condpred.experiments import e3

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/E3" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
CACHE = "E3/smoke" if SMOKE else "E3"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SPEC = e3.build_initial_spec(SMOKE)

## 2. Write spec

In [ ]:
report.write_spec(SPEC, OUT, modules=e3.E3_MODULES)
e3.verify_spec_immutable(OUT, SPEC, e3.E3_MODULES)

## 3. Leakage test

In [ ]:
leaks = pd.DataFrame(e3.run_leakage_e3(SPEC), columns=["pipeline", "pass", "note"])
report.save_table(leaks, OUT, "leakage")
assert leaks["pass"].all(), leaks[~leaks["pass"]]

## 4. Simulate clean paths (cached)

In [ ]:
e3.simulate_all(SPEC, CACHE)

## 5. Contaminate (cached per seed, all conditions)

In [ ]:
e3.contaminate_all(SPEC, CACHE)

## 6. E3a fidelity

In [ ]:
POOL = e3.make_pool()
E3A = e3.run_e3a(SPEC, OUT, CACHE, POOL)
display(E3A["fidelity"].query("system == 'ms_sv' and W == 120 and cond in ['J_l0.2_k5', 'N_w0.05']"))

## 7. E3b regime detection

In [ ]:
E3B = e3.run_e3b(SPEC, OUT, CACHE)
display(E3B["auc"].query("train_on == 'same' and W == 120").pivot_table(index="model", columns="cond", values="auc_pooled"))

## 8. E3d missing and stale data

In [ ]:
E3D = e3.run_e3d(SPEC, OUT, CACHE, POOL)
POOL.shutdown()
display(E3D["table"][["series", "pattern", "method", "rel_error_mean", "jaccard_mean", "drift_DET_mean", "drift_LAM_mean"]])

## 9. Hypothesis verdicts

In [ ]:
rows = e3.evaluate_e3a(SPEC, E3A) + e3.evaluate_e3b(SPEC, E3B) + e3.evaluate_e3d(SPEC, E3D) + [e3.e3c_row(SPEC)]
report.write_decision(rows, OUT)
display(pd.read_csv(OUT / "decision.csv"))

## 10. Figures and tables

In [ ]:
e3.make_figures(OUT, FIG)
for name in ["fidelity", "feature_drift", "regime_auc", "missing_fill", "decision"]:
    assert (OUT / f"{name}.csv").exists(), name
print(sorted(p.name for p in FIG.glob("*.png")))